# Restore and verify local research assets

Setup is explicit; existing assets are verified before reuse.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from pathlib import Path
import hashlib,json,os,shutil,tarfile
ROOT=TRACE_ROOT
# Supply directories explicitly when transferring to a new workstation.
# ARTIFACT_ROOT contains the original affectively/... player/model/data paths.
# ARCHIVE_ROOT contains checkpoint_check.tar.gz, task_baseline.tar.gz,
# max_support_baseline.tar.gz and random_baseline.tar.gz.
ARTIFACT_ROOT=None
ARCHIVE_ROOT=None
SOURCE=ROOT/'assets/upstream_source'
REFERENCE=ROOT/'outputs/reference'
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def safe_extract(archive_path,target):
    target=Path(target).resolve();target.mkdir(parents=True,exist_ok=True)
    with tarfile.open(archive_path) as archive:
        for member in archive.getmembers():
            dest=(target/member.name).resolve()
            if not dest.is_relative_to(target) or not (member.isfile() or member.isdir()):raise ValueError('Unsafe archive member: '+member.name)
            if member.isdir():dest.mkdir(parents=True,exist_ok=True);continue
            data=archive.extractfile(member).read()
            if dest.exists():
                if hashlib.sha256(dest.read_bytes()).digest()!=hashlib.sha256(data).digest():raise ValueError('Existing extracted file differs: '+str(dest))
            else:
                dest.parent.mkdir(parents=True,exist_ok=True)
                with dest.open('xb') as stream:stream.write(data)
                dest.chmod(member.mode & 0o777)
safe_extract(ROOT/'upstream/corrected_native_source.tar.gz',SOURCE)
count=0
for manifest_path in ['upstream/source_manifest.json','evidence/provenance/source_build_manifest.json','evidence/provenance/model_data_manifest.json','evidence/provenance/dataset_manifest.json']:
    for row in json.loads((ROOT/manifest_path).read_text()):
        destination=SOURCE/row['path']
        if not destination.exists():
            if ARTIFACT_ROOT is None:raise FileNotFoundError('Supply ARTIFACT_ROOT for '+row['path'])
            origin=Path(ARTIFACT_ROOT)/row['path'];assert sha(origin)==row['sha256']
            destination.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(origin,destination)
        assert sha(destination)==row['sha256'],destination
        count+=1
for row in json.loads((ROOT/'evidence/provenance/reference_archives.json').read_text()):
    destination=ROOT/row['path']
    if not destination.exists():
        if ARCHIVE_ROOT is None:raise FileNotFoundError('Supply ARCHIVE_ROOT for '+destination.name)
        origin=Path(ARCHIVE_ROOT)/destination.name;assert sha(origin)==row['sha256']
        destination.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(origin,destination)
    assert sha(destination)==row['sha256']
    safe_extract(destination,REFERENCE)
print('Verified',count,'frozen source/player/model/data files and four archives. Historical extraction:',REFERENCE.relative_to(ROOT))


Verified 274 frozen source/player/model/data files and four archives. Historical extraction: outputs/reference
